# Day 2 Exam: PyTorch & Data Pipelines

**End-of-day interview simulation** covering NB00-02 topics: PyTorch fundamentals, data pipelines, and embedding search.

---

### Rules
- **45 minutes total.** Set a timer now.
- No documentation, no autocomplete, no AI assistance.
- Write clean, working code from memory.
- Run all test cells to verify your solutions.

### Structure
- **Warmup** (3 problems, ~5 min each): Quick recall on core concepts.
- **Main Problems** (2 problems, ~30 min timer each): Deeper implementation challenges.

### Scoring
- All assertions pass = full credit per problem.
- Partial credit if the approach is correct but minor bugs remain.
- If you get stuck on a warmup for > 5 min, move on.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

---
## Warmup (5 min each)

Set a **5-minute timer** for each warmup problem. Move on if you exceed it.

### Warmup 1: Cosine Similarity

Implement `cosine_similarity(a, b)` from scratch (no `F.cosine_similarity`).

**Signature:**
```python
def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor
```

**Requirements:**
- `a` and `b` are 1-D tensors of the same length.
- Returns a scalar tensor: `dot(a, b) / (||a|| * ||b||)`.
- Handle the math yourself using `torch.dot`, `torch.norm`, or equivalent ops.

In [ ]:
# YOUR CODE HERE


In [ ]:
# --- Tests: Warmup 1 ---

# Identical vectors -> similarity 1.0
v = torch.tensor([1.0, 2.0, 3.0])
assert torch.isclose(cosine_similarity(v, v), torch.tensor(1.0), atol=1e-6), "Identical vectors should give 1.0"

# Orthogonal vectors -> similarity 0.0
a = torch.tensor([1.0, 0.0])
b = torch.tensor([0.0, 1.0])
assert torch.isclose(cosine_similarity(a, b), torch.tensor(0.0), atol=1e-6), "Orthogonal vectors should give 0.0"

# Opposite vectors -> similarity -1.0
a = torch.tensor([1.0, 2.0])
b = torch.tensor([-1.0, -2.0])
assert torch.isclose(cosine_similarity(a, b), torch.tensor(-1.0), atol=1e-6), "Opposite vectors should give -1.0"

# Normalized vectors: cosine similarity == dot product
a = torch.randn(128)
b = torch.randn(128)
a_norm = a / a.norm()
b_norm = b / b.norm()
cos_sim = cosine_similarity(a_norm, b_norm)
dot_prod = torch.dot(a_norm, b_norm)
assert torch.isclose(cos_sim, dot_prod, atol=1e-5), "For normalized vectors, cosine sim should equal dot product"

# Return type is a scalar tensor
assert cos_sim.dim() == 0, "Should return a scalar tensor"

print("Warmup 1 PASSED")

### Warmup 2: Linear Beta Schedule

Implement the linear noise schedule used in DDPM.

**Signature:**
```python
def linear_beta_schedule(timesteps: int) -> torch.Tensor
```

**Requirements:**
- Returns a 1-D tensor of length `timesteps`.
- Values linearly spaced from `1e-4` (start) to `0.02` (end), inclusive.
- Use `torch.linspace`.

In [ ]:
# YOUR CODE HERE


In [ ]:
# --- Tests: Warmup 2 ---

betas = linear_beta_schedule(1000)

# Shape
assert betas.shape == (1000,), f"Expected shape (1000,), got {betas.shape}"

# Endpoints
assert torch.isclose(betas[0], torch.tensor(1e-4), atol=1e-7), f"First beta should be 1e-4, got {betas[0]}"
assert torch.isclose(betas[-1], torch.tensor(0.02), atol=1e-7), f"Last beta should be 0.02, got {betas[-1]}"

# Monotonically increasing
diffs = betas[1:] - betas[:-1]
assert (diffs > 0).all(), "Betas should be strictly monotonically increasing"

# All positive
assert (betas > 0).all(), "All betas should be positive"

# Different timestep count
betas_small = linear_beta_schedule(10)
assert betas_small.shape == (10,), f"Expected shape (10,), got {betas_small.shape}"
assert torch.isclose(betas_small[0], torch.tensor(1e-4), atol=1e-7)
assert torch.isclose(betas_small[-1], torch.tensor(0.02), atol=1e-7)

print("Warmup 2 PASSED")

### Warmup 3: Fake Image Dataset

Implement a synthetic image dataset for testing data pipelines.

**Signature:**
```python
class FakeImageDataset(Dataset):
    def __init__(self, n: int, size: int): ...
    def __len__(self) -> int: ...
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]: ...
```

**Requirements:**
- `n`: number of samples. `size`: spatial dimension (height = width).
- `__getitem__` returns `(image_tensor, label)`.
- `image_tensor` shape: `(3, size, size)`, dtype `float32`, values in `[-1, 1]`.
- `label = idx % 10`.
- Generate random data on each call (no need to store all images in memory).

In [ ]:
# YOUR CODE HERE


In [ ]:
# --- Tests: Warmup 3 ---

ds = FakeImageDataset(n=100, size=64)

# Length
assert len(ds) == 100, f"Expected length 100, got {len(ds)}"

# Sample
img, label = ds[0]

# Shape
assert img.shape == (3, 64, 64), f"Expected shape (3, 64, 64), got {img.shape}"

# Dtype
assert img.dtype == torch.float32, f"Expected float32, got {img.dtype}"

# Value range
assert img.min() >= -1.0, f"Min value should be >= -1.0, got {img.min()}"
assert img.max() <= 1.0, f"Max value should be <= 1.0, got {img.max()}"

# Label logic
assert ds[0][1] == 0
assert ds[7][1] == 7
assert ds[13][1] == 3
assert ds[99][1] == 9

# Works with DataLoader
loader = DataLoader(ds, batch_size=8)
batch_img, batch_label = next(iter(loader))
assert batch_img.shape == (8, 3, 64, 64), f"Batch shape wrong: {batch_img.shape}"
assert batch_label.shape == (8,), f"Label batch shape wrong: {batch_label.shape}"

print("Warmup 3 PASSED")

---
## Main Problems (30 min each)

Set a **30-minute timer** for each problem. These are interview-depth implementations.

### Main Problem 1: Video Clip Dataset & Collation

Build a dataset and custom collate function for a video generation training pipeline.

#### Part A: `VideoClipDataset(Dataset)`

**Signature:**
```python
class VideoClipDataset(Dataset):
    def __init__(self, clip_metadata: list[dict], min_quality: float = 0.5): ...
    def __len__(self) -> int: ...
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, str]: ...
```

**Requirements:**
- `clip_metadata`: list of dicts, each with keys: `video_path` (str), `start_frame` (int), `end_frame` (int), `caption` (str), `quality_score` (float 0-1).
- In `__init__`, filter out any clips with `quality_score < min_quality`. Store only the passing clips.
- `__getitem__` returns `(frame_tensor, caption)` where:
  - `frame_tensor`: shape `(3, 256, 256)`, dtype `float32` (simulate by generating random data -- we don't need actual video I/O here).
  - `caption`: the caption string from the metadata.

#### Part B: `clip_collate_fn`

**Signature:**
```python
def clip_collate_fn(batch: list[tuple[torch.Tensor, str]]) -> dict:
```

**Requirements:**
- `batch` is a list of `(frame_tensor, caption)` tuples from the dataset.
- Returns a dict with:
  - `'frames'`: stacked tensor of shape `(B, 3, 256, 256)`.
  - `'captions'`: `list[str]` of the raw caption strings.
  - `'caption_lengths'`: `torch.Tensor` of shape `(B,)` with the character length of each caption.

In [ ]:
# YOUR CODE HERE


In [ ]:
# --- Tests: Main Problem 1 ---

# Synthetic metadata
metadata = [
    {"video_path": "/data/vid1.mp4", "start_frame": 0, "end_frame": 16, "caption": "a cat sitting on a mat", "quality_score": 0.9},
    {"video_path": "/data/vid2.mp4", "start_frame": 32, "end_frame": 48, "caption": "dog", "quality_score": 0.3},
    {"video_path": "/data/vid3.mp4", "start_frame": 0, "end_frame": 24, "caption": "a beautiful sunset over the ocean with waves crashing", "quality_score": 0.8},
    {"video_path": "/data/vid4.mp4", "start_frame": 10, "end_frame": 26, "caption": "person walking", "quality_score": 0.1},
    {"video_path": "/data/vid5.mp4", "start_frame": 5, "end_frame": 21, "caption": "fireworks at night", "quality_score": 0.7},
]

# --- Test filtering ---
ds = VideoClipDataset(metadata, min_quality=0.5)
assert len(ds) == 3, f"Expected 3 clips after filtering (quality >= 0.5), got {len(ds)}"

ds_strict = VideoClipDataset(metadata, min_quality=0.85)
assert len(ds_strict) == 1, f"Expected 1 clip with quality >= 0.85, got {len(ds_strict)}"

ds_all = VideoClipDataset(metadata, min_quality=0.0)
assert len(ds_all) == 5, f"Expected 5 clips with min_quality=0.0, got {len(ds_all)}"

# --- Test __getitem__ ---
frame, caption = ds[0]
assert frame.shape == (3, 256, 256), f"Expected frame shape (3, 256, 256), got {frame.shape}"
assert frame.dtype == torch.float32, f"Expected float32, got {frame.dtype}"
assert isinstance(caption, str), f"Caption should be a string, got {type(caption)}"
assert len(caption) > 0, "Caption should not be empty"

# --- Test collate function ---
batch = [ds[i] for i in range(len(ds))]
collated = clip_collate_fn(batch)

assert isinstance(collated, dict), "Collate should return a dict"
assert set(collated.keys()) == {"frames", "captions", "caption_lengths"}, f"Wrong keys: {collated.keys()}"

B = len(ds)
assert collated["frames"].shape == (B, 3, 256, 256), f"Expected frames shape ({B}, 3, 256, 256), got {collated['frames'].shape}"
assert isinstance(collated["captions"], list), "captions should be a list"
assert all(isinstance(c, str) for c in collated["captions"]), "All captions should be strings"
assert len(collated["captions"]) == B, f"Expected {B} captions, got {len(collated['captions'])}"
assert collated["caption_lengths"].shape == (B,), f"Expected caption_lengths shape ({B},), got {collated['caption_lengths'].shape}"

# Verify caption lengths are correct
for i, cap in enumerate(collated["captions"]):
    assert collated["caption_lengths"][i].item() == len(cap), (
        f"Caption length mismatch at index {i}: expected {len(cap)}, got {collated['caption_lengths'][i].item()}"
    )

# --- Test with DataLoader ---
loader = DataLoader(ds, batch_size=2, collate_fn=clip_collate_fn)
batch_out = next(iter(loader))
assert batch_out["frames"].shape == (2, 3, 256, 256), f"DataLoader batch frames shape wrong: {batch_out['frames'].shape}"
assert len(batch_out["captions"]) == 2
assert batch_out["caption_lengths"].shape == (2,)

print("Main Problem 1 PASSED")

### Main Problem 2: Embedding Index with Search & Filter

Build an in-memory embedding search index -- the kind you'd use for nearest-neighbor retrieval
in a content pipeline (e.g., finding similar video clips, deduplication, retrieval-augmented generation).

**Signature:**
```python
class EmbeddingIndex:
    def __init__(self, records: list[dict]): ...
    def add(self, records: list[dict]) -> None: ...
    def search(self, query_embedding: np.ndarray, top_k: int = 5) -> list[dict]: ...
    def search_with_filter(self, query_embedding: np.ndarray, filter_fn, top_k: int = 5) -> list[dict]: ...
```

**Requirements:**
- Each record dict has keys: `'id'` (str/int), `'embedding'` (np.ndarray, 1-D), `'metadata'` (dict).
- `add(records)` appends new records to the index.
- `search(query_embedding, top_k)` returns the top-k most similar records by **cosine similarity**.
  - Each result dict has keys: `'id'`, `'score'` (float, the cosine similarity), `'metadata'`.
  - Results are sorted by score descending (most similar first).
- `search_with_filter(query_embedding, filter_fn, top_k)` is the same as `search`, but only considers records where `filter_fn(record['metadata'])` returns `True`.

**Hints:**
- Use numpy for the math. Vectorized cosine similarity: normalize embeddings, then matrix-vector dot product.
- `np.argsort` or `np.argpartition` for top-k selection.

In [ ]:
# YOUR CODE HERE


In [ ]:
# --- Tests: Main Problem 2 ---

np.random.seed(42)
dim = 64

# Create records with known embeddings
def make_record(rid, embedding, category="default", quality=0.5):
    return {
        "id": rid,
        "embedding": embedding / np.linalg.norm(embedding),  # pre-normalize
        "metadata": {"category": category, "quality": quality},
    }

# Build a small index
base_vec = np.random.randn(dim).astype(np.float32)
records = [
    make_record("a", base_vec + np.random.randn(dim) * 0.01, category="cat", quality=0.9),   # very similar to base
    make_record("b", base_vec + np.random.randn(dim) * 0.1, category="cat", quality=0.4),    # similar
    make_record("c", base_vec + np.random.randn(dim) * 0.5, category="dog", quality=0.8),    # somewhat similar
    make_record("d", -base_vec, category="dog", quality=0.7),                                 # opposite
    make_record("e", np.random.randn(dim).astype(np.float32), category="bird", quality=0.6), # random
]

index = EmbeddingIndex(records)

# --- Test search ordering ---
query = base_vec / np.linalg.norm(base_vec)
results = index.search(query, top_k=3)

assert len(results) == 3, f"Expected 3 results, got {len(results)}"
assert results[0]["id"] == "a", f"Most similar should be 'a', got '{results[0]['id']}'"
assert results[0]["score"] > results[1]["score"] > results[2]["score"], "Results should be sorted by score descending"

# Each result has correct keys
for r in results:
    assert set(r.keys()) == {"id", "score", "metadata"}, f"Result keys wrong: {r.keys()}"

# --- Test scores for normalized embeddings are in [-1, 1] ---
all_results = index.search(query, top_k=5)
for r in all_results:
    assert -1.0 - 1e-6 <= r["score"] <= 1.0 + 1e-6, f"Score {r['score']} out of range [-1, 1]"

# The opposite vector should have a negative score
d_result = [r for r in all_results if r["id"] == "d"][0]
assert d_result["score"] < 0, f"Opposite vector should have negative score, got {d_result['score']}"

# --- Test add ---
new_record = make_record("f", base_vec + np.random.randn(dim) * 0.005, category="cat", quality=1.0)
index.add([new_record])
results_after_add = index.search(query, top_k=2)
result_ids = [r["id"] for r in results_after_add]
assert "f" in result_ids or "a" in result_ids, "After adding very similar vector, top results should include 'a' or 'f'"

# --- Test search_with_filter ---
# Only "dog" category
dog_results = index.search_with_filter(query, filter_fn=lambda m: m["category"] == "dog", top_k=5)
for r in dog_results:
    assert r["metadata"]["category"] == "dog", f"Filter should only return dogs, got {r['metadata']['category']}"

# Only high quality
hq_results = index.search_with_filter(query, filter_fn=lambda m: m["quality"] >= 0.7, top_k=10)
for r in hq_results:
    assert r["metadata"]["quality"] >= 0.7, f"Filter should only return quality >= 0.7, got {r['metadata']['quality']}"

# Filter that matches nothing
empty_results = index.search_with_filter(query, filter_fn=lambda m: m["category"] == "unicorn", top_k=5)
assert len(empty_results) == 0, f"Expected 0 results for non-matching filter, got {len(empty_results)}"

# top_k larger than filtered set
bird_results = index.search_with_filter(query, filter_fn=lambda m: m["category"] == "bird", top_k=10)
assert len(bird_results) == 1, f"Only 1 bird record exists, should return 1, got {len(bird_results)}"

print("Main Problem 2 PASSED")

---
## Done

Stop your timer. Review which problems you passed and which you struggled with.

**If you failed a problem:** don't look at solutions yet. Note what you were missing,
go back to the relevant notebook (NB00-02), study the concept, then retry tomorrow from scratch.

**Passing criteria:** All 5 problems pass on first attempt within time limits = ready to move on.